In [1]:
import lfox
import lfox.lattice as lat
import lfox.evolution.hmc as lhmc
import jax
import jax.numpy as jnp
import numpy as np

In [2]:
d = 4
L = 8

In [3]:
import os

jax.config.update('jax_threefry_partitionable', True)

os.environ['XLA_FLAGS'] = '--xla_force_host_platform_device_count=16'
jax.devices()

[CpuDevice(id=0),
 CpuDevice(id=1),
 CpuDevice(id=2),
 CpuDevice(id=3),
 CpuDevice(id=4),
 CpuDevice(id=5),
 CpuDevice(id=6),
 CpuDevice(id=7),
 CpuDevice(id=8),
 CpuDevice(id=9),
 CpuDevice(id=10),
 CpuDevice(id=11),
 CpuDevice(id=12),
 CpuDevice(id=13),
 CpuDevice(id=14),
 CpuDevice(id=15)]

In [4]:
MyLat = lat.SquareLattice(dims=((L,)*d))
MyLat

In [5]:
phi_field = lat.LatticeField(MyLat)
phi_field.field = np.ones_like(phi_field.field)  # Cold start

In [6]:
from jax.sharding import PositionalSharding
from jax.experimental import mesh_utils

sharding = PositionalSharding(mesh_utils.create_device_mesh((2,2,2,2)))
x = jax.random.normal(jax.random.PRNGKey(1), (16, 16, 16, 16))
y = jax.device_put(x, sharding)

In [7]:
@jax.jit
def scalar_action(phi, kappa, lamb):
    S = 0.0
    F = phi.field
    for ax in range(d):
        S -= 2 * kappa * jnp.sum(F * phi.nn_field(ax))
    phi2 = F**2
    S += jnp.sum(phi2)
    S += lamb * jnp.sum( (phi2-1)**2 )

    return S

In [8]:
%time scalar_action(phi_field, 0.16, 1.1689)

CPU times: user 46.4 ms, sys: 7.17 ms, total: 53.5 ms
Wall time: 45.1 ms


Array(-1146.8799, dtype=float32)

In [9]:
T = 14.4  # 9 ms
flops = (L**d) * (3 + 2*d) + (L**d - 1) * 3
print(flops, flops/T/1e9)  # GFlops

57341 3.982013888888889e-06


In [10]:
phi_shard = jax.device_put(phi_field, sharding)

In [11]:
%%time
scalar_action(phi_shard, 0.16, 1.1689).block_until_ready()

CPU times: user 109 ms, sys: 20.9 ms, total: 130 ms
Wall time: 86.4 ms


Array(-1146.8799, dtype=float32)

In [12]:
T = 4  # 9 ms
flops = (L**d) * (3 + 2*d) + (L**d - 1) * 3
print(flops, flops/T/1e9)  # GFlops

57341 1.433525e-05


In [13]:
class ScalarAction(lhmc.Action):
    @staticmethod
    def _Sjax(phi, kappa, lamb):
        return scalar_action(phi, kappa, lamb)

    def _compute_forces(self):
        self.grads = {
            'phi': jax.jit(jax.grad(scalar_action)),
        }

        def force_func(fields):
            return {'phi': self.grads['phi'](fields['phi'], self.params['kappa'], self.params['lamb']) }

        self.forces = force_func


In [14]:
def make_action(kappa, lamb=1.1689):
    return ScalarAction({'phi': phi_field}, params={'kappa': kappa, 'lamb': lamb})

@jax.jit
def mag(phi):
    return jnp.sum(phi.field)

def mag_obs(fields, params):
    m = mag(fields['phi'])
    m2 = m**2
    m4 = m**4

    return jnp.array([m, m2, m4])

In [15]:
%%time
all_k = [0.15, 0.16, 0.17, 0.175, 0.18, 0.185, 0.19, 0.195, 0.2, 0.21, 0.22]
#all_k = [0.15, 0.16]

all_mags = {}
for k in all_k:
    S = make_action(k)

    HMC = lhmc.HMCEvolver(
        action=S,
        seed=1586,
        integrator=lhmc.LeapfrogIntegrator(eps=0.01, Nstep=100),
        observables={'mag': [mag_obs, 10]}
    )

    for _ in range(100):
        HMC.evolve(warmup=True)
    for _ in range(300):
        HMC.evolve()

    all_mags[k] = HMC.obs_chain['mag']


CPU times: user 54.5 s, sys: 9.39 s, total: 1min 3s
Wall time: 49.5 s


In [17]:
print(np.mean(((np.array(all_mags[0.15])/(L**d)))[:,0]))
print(np.mean(((np.array(all_mags[0.16])/(L**d)))[:,0]))
print(np.mean(((np.array(all_mags[0.18])/(L**d)))[:,0]))
print(np.mean(((np.array(all_mags[0.20])/(L**d)))[:,0]))
print(np.mean(((np.array(all_mags[0.22])/(L**d)))[:,0]))

0.67683554
0.76611876
0.88563526
0.9620267
1.0233647


In [25]:
%%time

# Test timing on the HMC tutorial exercise 2.11
Lat6 = lat.SquareLattice(dims=((6,)*d))
phi6 = lat.LatticeField(Lat6)
phi6.field = np.ones_like(phi6.field)

S = ScalarAction({'phi': phi6}, params={'kappa': 0.185825, 'lamb': 1.1689})

HMC6 = lhmc.HMCEvolver(
    action=S,
    seed=195615,
    integrator=lhmc.LeapfrogIntegrator(eps=0.05, Nstep=20),
    observables={'mag': [mag_obs, 10]}
)

for _ in range(100):
    HMC6.evolve(warmup=True)

for _ in range(4900):
    HMC6.evolve()



CPU times: user 3.06 s, sys: 223 ms, total: 3.29 s
Wall time: 2.99 s


In [27]:
np.mean(np.array(HMC6.obs_chain['mag'])[:,0]/(6**4))

0.9117655